# 03 — Retrieval: Querying the Vector Store

In this notebook we will:
1. Connect to our existing ChromaDB collection
2. Explore semantic search (finding by meaning)
3. Use metadata filtering (narrowing by speaker, role, section)
4. Combine semantic search + metadata filters
5. Understand retrieval quality — what works, what doesn't

### Key concepts
- **Semantic search**: Find chunks whose *meaning* is close to the query, regardless of exact words used.
- **Metadata filtering**: Narrow the search space before semantic matching (e.g., only CFO, only Q&A section).
- **Distance**: How far a result is from the query in embedding space. Lower = more relevant. With cosine distance: 0.0 = identical, 2.0 = opposite.

**Important:** No LLM in this notebook — we're only looking at what the retriever brings back. This is the foundation that makes or breaks a RAG system.

## Setup

In [1]:
import chromadb
from chromadb.utils import embedding_functions

In [2]:
# Connect to our persistent ChromaDB
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

client = chromadb.PersistentClient(path="../chroma_db")
collection = client.get_collection(
    name="earnings_calls",
    embedding_function=embedding_fn,
)

print(f"Connected to collection: {collection.name}")
print(f"Total documents: {collection.count()}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Connected to collection: earnings_calls
Total documents: 75


In [3]:
def show_results(results, max_text=200):
    """Helper to display query results in a readable format."""
    docs = results["documents"][0]
    metas = results["metadatas"][0]
    dists = results["distances"][0]
    
    for i, (doc, meta, dist) in enumerate(zip(docs, metas, dists)):
        print(f"--- Result {i+1} (distance: {dist:.3f}) ---")
        print(f"  Speaker: {meta.get('speaker', '?')} | Role: {meta.get('role', '?')} | Section: {meta.get('section', '?')}")
        print(f"  Text: {doc[:max_text]}{'...' if len(doc) > max_text else ''}")
        print()

## Step 1: Pure semantic search

Let's start with basic queries — no filters, just meaning-based retrieval.

In [4]:
# Query 1: A straightforward financial question
results = collection.query(
    query_texts=["What were the revenue results for the quarter?"],
    n_results=3,
)
print("Query: 'What were the revenue results for the quarter?'\n")
show_results(results)

Query: 'What were the revenue results for the quarter?'

--- Result 1 (distance: 0.383) ---
  Speaker: Timothy Donald Cook | Role: Chief Executive Officer | Section: prepared_remarks
  Text: Thank you, Suhasini. Good afternoon, everyone, and thanks for joining the call. Before I talk about our results, I'd like to take a moment to acknowledge the devastating wildfires that impacted the Lo...

--- Result 2 (distance: 0.460) ---
  Speaker: Kevan Parekh | Role: Senior Vice President, Chief Financial Officer | Section: prepared_remarks
  Text: Thanks, Tim, and good afternoon, everyone. I'm going to cover the results for the first quarter of our fiscal year. We are very pleased to report an all-time high for revenue, with December quarter re...

--- Result 3 (distance: 0.482) ---
  Speaker: Timothy Donald Cook | Role: Chief Executive Officer | Section: q_and_a
  Text: You know, I don't want to project sales for the current quarter by region. But if you look at the channel inventory and look

In [5]:
# Query 2: A topic that could appear in multiple speakers' turns
results = collection.query(
    query_texts=["What is happening with Apple Intelligence?"],
    n_results=3,
)
print("Query: 'What is happening with Apple Intelligence?'\n")
show_results(results)

Query: 'What is happening with Apple Intelligence?'

--- Result 1 (distance: 0.357) ---
  Speaker: Richard Kramer | Role: Arete Research | Section: q_and_a
  Text: -- Analyst
Thanks very much. My first question is for Tim. I'd like to ask about what might accelerate the pace of Apple Intelligence adoption. I guess, do you see this simply as a question of time, i...

--- Result 2 (distance: 0.377) ---
  Speaker: Timothy Donald Cook | Role: Chief Executive Officer | Section: q_and_a
  Text: Yeah, Erik. Hi, it's Tim. The -- we did see that the markets where we had rolled out Apple Intelligence that the year over year performance on the iPhone 16 family was stronger than those where Apple ...

--- Result 3 (distance: 0.404) ---
  Speaker: Erik Woodring | Role: Analyst | Section: q_and_a
  Text: Great, guys. Thanks so much for taking my questions. You know, Tim, in your prepared remarks, you had noted that iPhone 16 models are selling better in markets where Apple Intelligence is available.

In [6]:
# Query 3: Something specific that might not have good matches
results = collection.query(
    query_texts=["What is the company's strategy for electric vehicles?"],
    n_results=3,
)
print("Query: 'What is the company's strategy for electric vehicles?'\n")
show_results(results)
print("^ Notice the high distances — the retriever still returns results,")
print("  but they're not actually relevant. This is a known RAG pitfall.")

Query: 'What is the company's strategy for electric vehicles?'

--- Result 1 (distance: 0.719) ---
  Speaker: Kevan Parekh | Role: Senior Vice President, Chief Financial Officer | Section: q_and_a
  Text: Yes, Amit, let me take that one. You know, as I mentioned in my remarks, we're guiding to 46.5% to 47.5 %. So, we think we're very pleased with that level of guidance. As you mentioned, there's always...

--- Result 2 (distance: 0.737) ---
  Speaker: Kevan Parekh | Role: Senior Vice President, Chief Financial Officer | Section: prepared_remarks
  Text: Thanks, Tim, and good afternoon, everyone. I'm going to cover the results for the first quarter of our fiscal year. We are very pleased to report an all-time high for revenue, with December quarter re...

--- Result 3 (distance: 0.748) ---
  Speaker: Timothy Donald Cook | Role: Chief Executive Officer | Section: q_and_a
  Text: I don't know the answer to your question precisely, but I think it is a combination of these products are so c

## Step 2: Metadata filtering with `where`

ChromaDB's `where` parameter lets you filter documents *before* semantic search.

This is powerful because it narrows the search to only relevant chunks.

In [7]:
# Only search in the CFO's statements
results = collection.query(
    query_texts=["What about gross margins?"],
    n_results=3,
    where={"role": "Senior Vice President, Chief Financial Officer"},
)
print("Query: 'What about gross margins?' (CFO only)\n")
show_results(results)

Query: 'What about gross margins?' (CFO only)

--- Result 1 (distance: 0.443) ---
  Speaker: Kevan Parekh | Role: Senior Vice President, Chief Financial Officer | Section: q_and_a
  Text: Yeah, thanks, Krish, for the question. So, on the product side, as you mentioned, you know, we had pretty strong sequential improvement, 300 basis points for the December quarter. You know, that was r...

--- Result 2 (distance: 0.450) ---
  Speaker: Kevan Parekh | Role: Senior Vice President, Chief Financial Officer | Section: q_and_a
  Text: Yes, Amit, let me take that one. You know, as I mentioned in my remarks, we're guiding to 46.5% to 47.5 %. So, we think we're very pleased with that level of guidance. As you mentioned, there's always...

--- Result 3 (distance: 0.465) ---
  Speaker: Kevan Parekh | Role: Senior Vice President, Chief Financial Officer | Section: q_and_a
  Text: Yeah, great. Hi, David, how are you? So, on the services gross margin, I think maybe stepping back a second, you know, s

In [8]:
# Only search in the Q&A section (analyst questions + executive answers)
results = collection.query(
    query_texts=["What is the outlook for China?"],
    n_results=3,
    where={"section": "q_and_a"},
)
print("Query: 'What is the outlook for China?' (Q&A section only)\n")
show_results(results)

Query: 'What is the outlook for China?' (Q&A section only)

--- Result 1 (distance: 0.445) ---
  Speaker: Erik Woodring | Role: Analyst | Section: q_and_a
  Text: OK. Thank you for that, Tim. It's helpful. And then, you know, if we just touch on China, obviously, in the news fairly frequently.
If we set aside China macro, which I understand is still challenging...

--- Result 2 (distance: 0.501) ---
  Speaker: Timothy Donald Cook | Role: Chief Executive Officer | Section: q_and_a
  Text: Yeah, sure. If you look at our greater China revenue for the quarter, we were down 11% year over year. And over half of the decline that we experienced was driven by change in channel inventory from t...

--- Result 3 (distance: 0.568) ---
  Speaker: Wamsi Mohan | Role: Analyst | Section: q_and_a
  Text: Yes, thank you so much. Tim, I want to follow up on your comment about channel inventory in China. I was wondering if you could maybe address more broadly channel inventory across your different produ.

## Step 3: Compound filters

ChromaDB supports `$and` and `$or` operators to combine multiple conditions.

---

Write a query that finds what the **CEO** said about **emerging markets**, but only in the **prepared_remarks** section.

ChromaDB uses `$and` to combine filters. Each condition is a dict with a single field.

In [12]:
results_compound = collection.query(
    query_texts=["emerging markets growth"],
    n_results=3,
    where={
        "$and": [
            {"role": "Chief Executive Officer"},
            {"section": "prepared_remarks"}
        ]
    },
)
print("Query: 'emerging markets growth' (CEO + prepared_remarks only)\n")
show_results(results_compound)

Query: 'emerging markets growth' (CEO + prepared_remarks only)

--- Result 1 (distance: 0.601) ---
  Speaker: Timothy Donald Cook | Role: Chief Executive Officer | Section: prepared_remarks
  Text: Thank you, Suhasini. Good afternoon, everyone, and thanks for joining the call. Before I talk about our results, I'd like to take a moment to acknowledge the devastating wildfires that impacted the Lo...



## Step 4: Understanding retrieval quality

Let's compare the same question with and without filters to see how filtering changes result quality.

In [14]:
query = "What is the guidance for next quarter's gross margin?"

# Without filter
results_all = collection.query(query_texts=[query], n_results=3)
print("WITHOUT filter:")
print("=" * 50)
show_results(results_all, max_text=120)

# With CFO filter
results_cfo = collection.query(
    query_texts=[query],
    n_results=3,
    where={"role": "Senior Vice President, Chief Financial Officer"},
)
print("WITH CFO filter:")
print("=" * 50)
show_results(results_cfo, max_text=120)

WITHOUT filter:
--- Result 1 (distance: 0.385) ---
  Speaker: Amit Daryanani | Role: Analyst | Section: q_and_a
  Text: Perfect. Thank you. And then, I guess, just a question on gross margins for the March quarter. You folks are guiding gro...

--- Result 2 (distance: 0.389) ---
  Speaker: David Vogt | Role: Analyst | Section: q_and_a
  Text: Great. Thanks guys for taking my question. So, maybe, Tim, this is for you. I'm trying to think about your commentary ar...

--- Result 3 (distance: 0.397) ---
  Speaker: Kevan Parekh | Role: Senior Vice President, Chief Financial Officer | Section: q_and_a
  Text: Yes, Amit, let me take that one. You know, as I mentioned in my remarks, we're guiding to 46.5% to 47.5 %. So, we think ...

WITH CFO filter:
--- Result 1 (distance: 0.397) ---
  Speaker: Kevan Parekh | Role: Senior Vice President, Chief Financial Officer | Section: q_and_a
  Text: Yes, Amit, let me take that one. You know, as I mentioned in my remarks, we're guiding to 46.5% to 47.5 %.

In [15]:
# Distance comparison: are filtered results actually closer?
print("Distance comparison:")
print(f"  Without filter: {[f'{d:.3f}' for d in results_all['distances'][0]]}")
print(f"  With CFO filter: {[f'{d:.3f}' for d in results_cfo['distances'][0]]}")
print()
print("Note: filtered results might have HIGHER distances because we")
print("excluded some semantically close chunks. But the results are more")
print("*useful* because they come from the right person (the CFO).")
print("This is the tradeoff: relevance vs. precision.")

Distance comparison:
  Without filter: ['0.385', '0.389', '0.397']
  With CFO filter: ['0.397', '0.412', '0.505']

Note: filtered results might have HIGHER distances because we
excluded some semantically close chunks. But the results are more
*useful* because they come from the right person (the CFO).
This is the tradeoff: relevance vs. precision.


## Step 5: Exploring the collection

Useful queries to understand what's in your vector store.

In [16]:
# Get all documents to explore metadata
all_docs = collection.get(include=["metadatas"])

# What speakers do we have?
speakers = set(m["speaker"] for m in all_docs["metadatas"])
print(f"Unique speakers ({len(speakers)}):")
for s in sorted(speakers):
    count = sum(1 for m in all_docs["metadatas"] if m["speaker"] == s)
    print(f"  {s}: {count} turns")

print(f"\nSections:")
sections = set(m.get("section", "unknown") for m in all_docs["metadatas"])
for s in sorted(sections):
    count = sum(1 for m in all_docs["metadatas"] if m.get("section") == s)
    print(f"  {s}: {count} turns")

Unique speakers (15):
  Amit Daryanani: 2 turns
  Atif Malik: 2 turns
  Ben Bollin: 1 turns
  Ben Reitzes: 3 turns
  Benjamin Bollin: 2 turns
  David Vogt: 2 turns
  Erik Woodring: 4 turns
  Kevan Parekh: 7 turns
  Krish Sankar: 3 turns
  Mike Ng: 4 turns
  Richard Kramer: 3 turns
  Samik Chatterjee: 3 turns
  Suhasini Chandramouli: 13 turns
  Timothy Donald Cook: 23 turns
  Wamsi Mohan: 3 turns

Sections:
  prepared_remarks: 4 turns
  q_and_a: 71 turns


## Summary

In this notebook you learned:
- Pure semantic search finds by meaning, but always returns results — even irrelevant ones (watch the distances!)
- Metadata filtering narrows the search space to specific speakers, roles, or sections
- Compound filters (`$and`) let you combine multiple conditions
- Filtering can improve *usefulness* even when distances are higher
- The retriever is the foundation — if it brings back bad chunks, no LLM can fix that

**Next step:** In notebook 04 we'll connect an LLM (Groq) and build the full RAG pipeline — retrieval + generation in one chain.